# 🔌 ElectroGyaan AI — Energy Consumption Analysis

## 🎯 Objective
- Detect anomalous energy usage
- Predict future consumption patterns

## 📂 Dataset Features
- flat_id
- timestamp
- units_kWh
- is_anomaly (ground truth)

## 🧠 Approach
1. Data Loading
2. Preprocessing
3. Feature Engineering
4. Exploratory Data Analysis (EDA)
5. Model Training
6. Model Evaluation
7. Model Export

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, mean_absolute_error, r2_score

import joblib

plt.style.use("ggplot")

In [ ]:
df = pd.read_csv("../data/electrogyaan_dataset.csv")
df.head()

## 🔍 Data Inspection

We examine:
- Data types
- Missing values
- Basic statistics

In [ ]:
df.info()
df.describe()

## 🧹 Data Preprocessing

- Convert timestamp to datetime
- Convert anomaly label to numeric

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["is_anomaly"] = df["is_anomaly"].astype(int)

## ⚙️ Feature Engineering

We extract time-based features and apply cyclical encoding:
- Hour of day
- hour_sin and hour_cos to capture cyclic nature of time

In [ ]:
df["hour"] = df["timestamp"].dt.hour

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df.head()

## 📊 Distribution of Energy Consumption

In [ ]:
sns.histplot(df["units_kWh"], bins=50)
plt.title("Distribution of Energy Consumption")
plt.show()

## 📦 Outlier Detection (Boxplot)

In [ ]:
sns.boxplot(x=df["units_kWh"])
plt.title("Boxplot of Energy Consumption")
plt.show()

## 📈 Hourly Energy Usage Trend

In [ ]:
hourly_avg = df.groupby("hour")["units_kWh"].mean()

hourly_avg.plot()
plt.title("Average Consumption by Hour")
plt.xlabel("Hour")
plt.ylabel("kWh")
plt.show()

## ✂️ Train-Test Split

We use time-based splitting to avoid data leakage.

In [ ]:
df = df.sort_values("timestamp")

split = int(0.8 * len(df))

train = df.iloc[:split]
test = df.iloc[split:]

## 🚨 Anomaly Detection using Isolation Forest

In [ ]:
features = ["hour_sin", "hour_cos", "units_kWh"]

iso_model = IsolationForest(contamination=0.05, random_state=42)
iso_model.fit(train[features])

In [ ]:
test["pred"] = iso_model.predict(test[features])
test["pred"] = test["pred"].map({1: 0, -1: 1})

print(classification_report(test["is_anomaly"], test["pred"]))

## 🔮 Energy Consumption Prediction using Linear Regression

In [ ]:
X_train = train[["hour_sin", "hour_cos"]]
y_train = train["units_kWh"]

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

In [ ]:
X_test = test[["hour_sin", "hour_cos"]]
y_test = test["units_kWh"]

preds = lr_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, preds))
print("R2 Score:", r2_score(y_test, preds))

## 💾 Save Trained Models

These models will be used in FastAPI service.

In [ ]:
joblib.dump(iso_model, "../models/iso_model.pkl")
joblib.dump(lr_model, "../models/lr_model.pkl")

## 💡 Key Insights

- Energy usage peaks during morning and evening hours
- Majority of consumption values are within normal range
- Around 5% of readings are anomalies (spikes)
- Isolation Forest effectively detects abnormal patterns
- Linear Regression captures hourly trends reasonably well

## ⚠️ Limitations

- Synthetic dataset assumptions
- No weather or occupancy data
- Simple linear model used for prediction

## 🚀 Next Steps

- Deploy models via FastAPI
- Build real-time ingestion pipeline
- Visualize insights on dashboard